# 07 · The agent and the app

The right-hand side of the architecture: the chat agent's tools, the tool-calling loop that drives them,
and the FastAPI app around both. Nothing here needs Databricks. The app ships with a **fake data layer**
(`webapp/_fake.py`, the same one `NW_FAKE_DATA=1` turns on) that stands in for Lakebase, Unity Catalog
and the model, and this notebook drives the *real* loop and the *real* routes against it.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent   # run from the repo root or tutorials/
sys.path[:0] = [str(ROOT), str(ROOT / "tutorials")]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import _data
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#898781"])})
pd.set_option("display.width", 160, "display.max_columns", 30)
print("data from:", _data.source())

import inspect
import json
import os
import re
from types import SimpleNamespace

from IPython.display import Markdown, display

from src import agent_chat, agent_tools
real_run_chat = agent_chat.run_chat          # keep the real loop; the fake layer replaces it with a script
real_dispatch = dict(agent_chat._DISPATCH)    # and the real (logged) tools

data from: local exports


## A tool is a plain function

Each tool is an ordinary Python function in `src/agent_tools.py` that returns a dict, `{"ok": True, ...}`
or `{"ok": False, "error": "..."}`, instead of raising: a structured error is something the model can
explain to the user, a traceback isn't. Here is one, as written:

In [2]:
print(inspect.getsource(agent_tools.get_area_summary))

@_logged("get_area_summary")
def get_area_summary(community_area: int) -> dict[str, Any]:
    """Plain-language summary of what's been happening in an area lately.

    Written nightly by notebook 09 (`ai_query` over the area's recent numbers), so this is a
    lookup, not a live LLM call. `context` holds the facts the narrative was written from; quote
    numbers from there.

    Returns:
        {"ok": True, "community_area": int, "area_name": str, "as_of_date": str,
         "narrative": str, "context": dict, "generated_at": str, "model": str}
        or {"ok": False, "error": str}
    """
    err = ad.validate_community_area(community_area)
    if err:
        return {"ok": False, "error": err}
    try:
        rows = ad.query_uc_cached(f"""
            SELECT community_area, as_of_date, narrative, context_json, model, generated_at
            FROM {ad.UC_NARRATIVES}
        """)
    except Exception as exc:
        return {"ok": False, "error": f"narratives table unavailable (has 

The model sees a JSON schema for it (`TOOL_SPECS` in `src/agent_chat.py`, the OpenAI function-calling
format that Databricks Foundation Model APIs speak), and the loop finds the function through `_DISPATCH`.
`tests/test_agent_wiring.py` checks that the three agree.

In [3]:
spec = next(s for s in agent_chat.TOOL_SPECS if s["function"]["name"] == "get_area_summary")
print(json.dumps(spec, indent=2))
print("dispatches to:", agent_chat._DISPATCH["get_area_summary"].__qualname__)

{
  "type": "function",
  "function": {
    "name": "get_area_summary",
    "description": "A short plain-language summary of what's been happening in a community area lately (recent crime and 311 patterns, hotspot blocks, rising signals), with the structured facts it was written from. Good first call for 'what's going on in X?'.",
    "parameters": {
      "type": "object",
      "properties": {
        "community_area": {
          "type": "integer",
          "description": "Chicago community area number, 1-77."
        }
      },
      "required": [
        "community_area"
      ]
    }
  }
}
dispatches to: get_area_summary


## `@_logged`: every call is recorded

The decorator binds the call's arguments to the tool's signature (so `user_id` and `community_area` are
found however they're passed), runs the tool, and writes a row to Lakebase's `tool_invocations`. Logging
is best-effort: it swallows its own failures, so it can never break the tool call. Change Data Feed
carries those rows to Unity Catalog, where `tool_usage_daily` (notebook `07`) reports requests and success
rates per tool. The original function stays reachable as `__wrapped__`, which the Insights page uses so a
page load doesn't count as agent usage.

In [4]:
print(inspect.getsource(agent_tools._logged))

def _logged(tool_name: str) -> Callable[[Callable[..., dict[str, Any]]], Callable[..., dict[str, Any]]]:
    """Log every call to the wrapped tool via `_log_invocation`.

    Arguments are bound to the tool's own signature, so `user_id` and `community_area` are found
    whether they're passed by position or keyword. Tools without a `user_id` log it as None.
    The undecorated function stays reachable as `__wrapped__`, for calls that shouldn't count as
    agent usage.
    """
    def decorator(fn: Callable[..., dict[str, Any]]) -> Callable[..., dict[str, Any]]:
        sig = inspect.signature(fn)

        @functools.wraps(fn)
        def wrapper(*args: Any, **kwargs: Any) -> dict[str, Any]:
            result = fn(*args, **kwargs)
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()
            _log_invocation(
                tool_name,
                result,
                user_id=bound.arguments.get("user_id"),
                community_area=bound.arg

## The fake data layer

`_fake.install()` swaps every Lakebase, Unity Catalog and LLM call for deterministic canned data, and
`tool_invocations` writes for a print. The tools themselves are the real ones wherever they're pure
logic over those reads (ranking, metric-name resolution, shaping).

In [5]:
from webapp import _fake

_fake.LATENCY_S = 0          # the fake normally sleeps so the UI's loading states show
_fake.install()

trend = agent_tools.get_area_trend(24, months=3)
m = trend["metrics"]["crime_total"]
print(f"{trend['area_name']}: {m['window_count']} crimes in the last 30 days, "
      f"{m['pct_change_vs_prior_window']:+.1f}% vs the 30 before, {m['pct_vs_normal']:+.1f}% vs normal")
ranked = agent_tools.rank_areas("violent", sort_by="above_normal", limit=3)       # a loose metric name
print("resolved metric:", ranked["metric"])
pd.DataFrame(ranked["areas"])[["rank", "area_name", "window_count", "normal_count", "pct_vs_normal"]]

West Town: 545 crimes in the last 30 days, +14.0% vs the 30 before, +21.7% vs normal


[fake] tool_invocations row: rank_areas area=None ok=True
resolved metric: crime_violent


,rank,area_name,window_count,normal_count,pct_vs_normal
0,1,Irving Park,154,92.2,67.0
1,2,Washington Heights,138,86.1,60.3
2,3,West Pullman,74,50.0,48.0


Note the `[fake] tool_invocations row` lines: that's `@_logged` at work. And `"violent"` became
`crime_violent`: `resolve_metric` maps loose names onto real ones, because left to guess, the model tries
plausible but wrong names.

## The real loop, with a scripted model

`agent_chat.run_chat` is the whole agent: send the system prompt, the conversation and the tool specs to
the model; run whatever tools it calls; append the results; repeat until it answers (at most four
rounds). To run it without a model endpoint, swap `_get_client` for a stand-in that speaks the same
OpenAI client interface. This one "decides" like a model would: on a fresh question it asks for two
tools, and once it has their results it writes an answer from them.

In [6]:
def message(content=None, calls=()):
    tool_calls = [SimpleNamespace(id=f"call_{i}", function=SimpleNamespace(name=n, arguments=json.dumps(a)))
                  for i, (n, a) in enumerate(calls)]
    return SimpleNamespace(choices=[SimpleNamespace(message=SimpleNamespace(content=content, tool_calls=tool_calls or None))])

class ScriptedModel:
    def __init__(self):
        self.requests = []
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self.create))

    def create(self, model, messages, tools):
        self.requests.append(messages)
        if messages[-1]["role"] == "user":                         # first round: look things up
            return message(calls=[("get_area_summary", {"community_area": 24}),
                                  ("get_next_week_outlook", {"community_area": 24})])
        results = {m["tool_call_id"]: json.loads(m["content"]) for m in messages if m["role"] == "tool"}
        outlook = results["call_1"]["area"]
        return message(f"**{outlook['area_name']}** next week: the forecast calls it **{outlook['call']}** "
                       f"(P(above normal) = {outlook['p_above_normal']}). "
                       f"Of that, {outlook['citywide_part_pct']:+.1f}% is citywide and {outlook['local_part_pct']:+.1f}% "
                       "is the area's own lean.")

model = ScriptedModel()
agent_chat._get_client = lambda: model
# _DISPATCH captured the tool functions when agent_chat was imported; point it at the faked ones
agent_chat._DISPATCH = {name: getattr(agent_tools, name) for name in agent_chat._DISPATCH}
trace = {}
new = real_run_chat([{"role": "user", "content": "How's West Town looking this week?"}],
                    current_user_id=1, active_area=24, trace=trace)
for m in new:
    calls = [c["function"]["name"] for c in m.get("tool_calls", [])]
    print(f"{m['role']:9s}", calls or (m["content"][:110] + ("..." if len(m["content"]) > 110 else "")))

assistant ['get_area_summary', 'get_next_week_outlook']
tool      {"ok": true, "community_area": 24, "area_name": "West Town", "as_of_date": "2026-10-01", "narrative": "(fake m...
tool      {"ok": true, "week_start": "2026-09-28", "week_end": "2026-10-04", "data_through": "2026-09-23", "citywide": {...
assistant **West Town** next week: the forecast calls it **unclear** (P(above normal) = 0.41). Of that, +4.2% is citywid...


The model saw a system prompt assembled for this user and this view: the base instructions, the logged-in
user (so it can fill `user_id` on the write tools without asking), the real metric names, and the area
open in the app (so "here" resolves):

In [7]:
system = model.requests[0][0]["content"]
print(f"{len(system):,} characters; it ends with:\n")
print("..." + system[-420:])

6,051 characters; it ends with:

...lable to the tools (use these exact strings): 311_abandoned_vehicle, 311_graffiti, 311_illegal_dumping, 311_rodent_rat, 311_streetlight_out, 311_total, 311_vacant_abandoned_building, crime_drugs, crime_fraud_financial, crime_other, crime_property, crime_public_order, crime_total, crime_violent, crime_weapons. The user is currently viewing West Town (community area 24) in the app; 'here' or 'this area' means that one.


And `trace` is what the `/chat` route writes to Lakebase's `chat_turns`, one row per turn, for analytics
and evaluation (notebook `07` turns it into `chat_turns_daily` and `chat_eval_set`):

In [8]:
print(json.dumps(trace, indent=2, default=str))
display(Markdown(new[-1]["content"]))

{
  "model": "databricks-claude-sonnet-4-5",
  "tool_calls": [
    {
      "name": "get_area_summary",
      "arguments": "{\"community_area\": 24}",
      "ok": true,
      "error": null,
      "latency_ms": 0
    },
    {
      "name": "get_next_week_outlook",
      "arguments": "{\"community_area\": 24}",
      "ok": true,
      "error": null,
      "latency_ms": 1
    }
  ],
  "latency_ms": 1
}


**West Town** next week: the forecast calls it **unclear** (P(above normal) = 0.41). Of that, +4.2% is citywide and -7.0% is the area's own lean.

## Adding a tool

Four steps, in `src/agent_tools.py` and `src/agent_chat.py`:
1. Write the function and decorate it with `@_logged("name")`; return `{"ok": ...}` dicts.
2. Describe it in `TOOL_SPECS` (what it's for, and each parameter).
3. Add it to `_DISPATCH`.
4. Tell the model when to use it in `SYSTEM_PROMPT`, and run `tests/test_agent_wiring.py`.

Here's a toy one, wired in for this session only, and the same check the wiring test runs:

In [9]:
from src.community_areas import AREA_NAMES, POPULATION

@agent_tools._logged("get_area_population")
def get_area_population(community_area: int) -> dict:
    """Residents of a community area (ACS 2023)."""
    if community_area not in POPULATION:
        return {"ok": False, "error": f"community_area must be 1-77, got {community_area}"}
    return {"ok": True, "area_name": AREA_NAMES[community_area], "population": POPULATION[community_area]}

spec = {"type": "function", "function": {
    "name": "get_area_population", "description": "How many people live in a Chicago community area.",
    "parameters": {"type": "object", "required": ["community_area"],
                   "properties": {"community_area": {"type": "integer", "description": "1-77"}}}}}
specs, dispatch = agent_chat.TOOL_SPECS + [spec], {**real_dispatch, "get_area_population": get_area_population}

for s in specs:                       # what tests/test_agent_wiring.py checks for every tool
    fn, params = dispatch[s["function"]["name"]], inspect.signature(dispatch[s["function"]["name"]]).parameters
    assert hasattr(fn, "__wrapped__") and set(s["function"]["parameters"]["properties"]) <= set(params)
print("wiring OK for", len(specs), "tools;", get_area_population(24))

[fake] tool_invocations row: get_area_population area=24 ok=True
wiring OK for 11 tools; {'ok': True, 'area_name': 'West Town', 'population': 86453}


## The app

`webapp/main.py` is FastAPI with Jinja templates. Most routes return an **HTML fragment** that HTMX swaps
into the page; the map and charts fetch small JSON endpoints and draw client-side. FastAPI's test client
can drive the real routes against the fake data, the same way the fake-mode dev server does:

In [10]:
os.environ.update(NW_FAKE_DATA="1", SESSION_SECRET="tutorial")
from fastapi.testclient import TestClient
from webapp.main import app

client = TestClient(app)
client.post("/signup", data={"email": "tutorial@example.com", "display_name": "Tutorial"})   # fake users table
panel = client.get("/area-detail", params={"community_area": 24})
lower = client.get("/area-insight", params={"community_area": 24})
print(f"/area-detail -> {panel.status_code}, {len(panel.text):,} bytes of HTML, cards: "
      f"{re.findall(r'<h3[^>]*>([^<]+)', panel.text)}")
print(f"/area-insight -> {lower.status_code}, cards: {re.findall(r'<h3[^>]*>([^<]+)', lower.text)}")

layer = client.get("/api/map-data", params={"metric": "crime_total", "mode": "normal"}).json()
print(f"/api/map-data -> {len(layer['data'])} areas, e.g. area 24: {layer['data']['24']}")

/area-detail -> 200, 37,584 bytes of HTML, cards: ['Crime by type', 'Rolling 12-month change']
/area-insight -> 200, cards: ['The week ahead', 'Upcoming events', "What's happening here", 'Signals to watch']
/api/map-data -> 77 areas, e.g. area 24: {'window_count': 545, 'normal_count': 447.9, 'pct_vs_normal': 21.7}


The area panel and its lower cards are separate requests on purpose: the lower cards read Unity Catalog
through the SQL warehouse, which can take seconds when it's cold, so the panel renders first and the rest
fills in.

To see it all in a browser with the same fake data:

```bash
NW_FAKE_DATA=1 SESSION_SECRET=local-dev-only uvicorn webapp.main:app --reload
```

## Where the series leaves you

- **01** the layers and the lineage, **02** ingestion and silver, **03** the trend numbers, **04** the
  311 lead-lag analysis, **05** the forecast and how to improve it, **06** the events model, **07** the
  agent and the app.
- `docs/architecture.md`, `docs/modeling.md` and `docs/operations.md` are the reference versions of all
  of this; `tests/` pins down the behavior of every model and shaping function you've seen.